# Chapter 8 — The Shape of an Embedding Space

**Book alignment:** Embeddings From First Principles, Chapter 8

**Notebook role:** `ARTIFACT_REPLAY` — reads the frozen experiment artifact(s) this chapter cites and re-derives the numbers quoted in the prose (assertions fail if the artifact drifts).

**Question this notebook isolates:** The standard "repair" for raw embeddings —
mean-centre, drop the top principal components, rescale to equal variance (whitening) —
drives the similarity origin toward zero. Does it *help* retrieval? On RELATE's
contrastively-trained encoders the committed measurement says the signed gain is
**non-positive**: BERT-whitening costs ~4 nDCG@10 points and naive full whitening is
catastrophic.

In [ ]:
from pathlib import Path
import json
import numpy as np

rng = np.random.default_rng(0)


def find_repo_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "experiments" / "embeddings-from-first-principles" / "wave1").is_dir():
            return c
    raise RuntimeError("run from a checkout containing experiments/embeddings-from-first-principles")


ROOT = find_repo_root(Path.cwd().resolve())
EXP = ROOT / "experiments" / "embeddings-from-first-principles"


def art(wave: str, name: str) -> dict:
    return json.loads((EXP / wave / "artifacts" / name).read_text())

## 1. The whitening pipeline drives the similarity origin to zero — reproduce it

In [ ]:
n, d = 400, 64
Z = rng.standard_normal((n, d))
Z[:, 0] += 8.0                                        # a dominant offset direction
Z = Z / np.linalg.norm(Z, axis=1, keepdims=True)

def origin(M):
    Mn = M / np.linalg.norm(M, axis=1, keepdims=True)
    C = Mn @ Mn.T
    return float(C[np.triu_indices(n, 1)].mean())

W = Z - Z.mean(0)                                     # 1. centre
U, S, Vt = np.linalg.svd(W - W.mean(0), full_matrices=False)
W2 = W - (W @ Vt[:3].T) @ Vt[:3]                      # 2. drop top-3 PCs
Uw, Sw, Vtw = np.linalg.svd(W2, full_matrices=False)
W3 = (W2 @ Vtw.T) / (Sw + 1e-6) @ Vtw                 # 3. rescale to equal variance

print(f"origin  raw={origin(Z):+.3f}  centred={origin(W):+.3f}"
      f"  drop-3={origin(W2):+.3f}  whitened={origin(W3):+.3f}")
assert abs(origin(W3)) < abs(origin(Z))

## 2. The signed whitening gain — measured on RELATE (Wave 2)

In [ ]:
wg = art("wave2", "whitening-gain.json")["models"]["bge-large"]["stages"]
for stage in ("raw", "centered", "drop_top3", "whitened_top95var", "whitened_full"):
    s = wg[stage]
    print(f"  {stage:20} nDCG@10={s['ndcg10']:.4f}  hard-neg margin={s['hard_negative_margin']:+.3f}"
          f"  origin={s['mean_random_cosine']:+.4f}")

# whitening HURTS a model whose raw geometry is already near-isotropic
assert wg["whitened_top95var"]["ndcg10"] < wg["raw"]["ndcg10"] - 0.03
assert wg["whitened_full"]["ndcg10"] < 0.2                       # naive full whitening: catastrophic
# centring alone is neutral; hard-negative margin barely moves either way
assert abs(wg["centered"]["ndcg10"] - wg["raw"]["ndcg10"]) < 0.005
print("\nsigned gain <= 0: this model's raw geometry carried no nuisance bulk to remove")
print("shape repair never adds a distinction the model did not encode")

## What we earned

Whitening / centring is a *nuisance-removal* operation, and the **sign** of its effect
diagnoses the raw geometry: a large positive gain means heavy nuisance structure was
present; a non-positive gain (all five RELATE encoders) means the geometry was already
clean and the rescaling only amplified low-variance noise — naive full whitening divides
~380 near-zero-variance axes by ≈0 and collapses retrieval to nDCG@10 0.07. Describe a
model by the geometry it produces, then measure before repairing it.

**Notebook 09 / Chapter 9** implements retrieval from first principles — embed, score, sort
— with no vector database, so every parameter is visible.